# IID `fieldgen2d_sva` with the structured Freyberg model

This notebook demonstrates the *iid* (independent-and-identically-distributed) variant
of `fieldgen2d_sva` that was added to `pypestutils`. The standard `fieldgen2d_sva`
draws its underlying standard-normal variates internally, seeded by
`initialize_randgen`. The iid variant — `fieldgen2d_sva_iid` — instead accepts the
standard-normal variates from the caller, which:

- decouples random-field generation from the Fortran library's random number generator,
- lets you drive the field with any RNG (e.g. NumPy's `default_rng`),
- exposes the fact that, for `transtype=0` and a zero mean, the resulting field is a
  *linear* function of the input variates — useful for sensitivity analysis,
  ensemble post-processing, and reproducibility across runs/machines.

`fill_stdnormal` is also exposed so you can reproduce the exact draw sequence the
non-iid version would have used.

In [ ]:
import os
import shutil

import flopy
import matplotlib.pyplot as plt
import numpy as np

from pypestutils.pestutilslib import PestUtilsLib

## Setup the model directory

In [ ]:
org_d = "freyberg_monthly"
w_d = "freyberg_iid_fieldgen"
if os.path.exists(w_d):
    shutil.rmtree(w_d)
shutil.copytree(org_d, w_d)

In [ ]:
sim = flopy.mf6.MFSimulation.load(sim_ws=w_d)
m = sim.get_model()

## Install the MF6 grid and get cell centroids

In [ ]:
lib = PestUtilsLib()
grid_info = lib.install_mf6_grid_from_file(
    "structgrid", os.path.join(w_d, "freyberg6.dis.grb")
)
grid_info

In [ ]:
nrow = grid_info["ndim2"]
ncol = grid_info["ndim1"]
nlay = grid_info["ndim3"]

In [ ]:
easting, northing, elev = lib.get_cell_centres_mf6("structgrid", grid_info["ncells"])
# 2D problem -> use just the first layer's nodes
easting = easting[: nrow * ncol]
northing = northing[: nrow * ncol]
Easting = easting.reshape((nrow, ncol))
Northing = northing.reshape((nrow, ncol))
easting.shape

## Build the per-node variogram property arrays

These are the same inputs that `fieldgen2d_sva` takes — mean, variance, range (`aa`),
anisotropy ratio, and bearing. They can be scalars or per-node arrays.

In [ ]:
area = np.ones_like(easting)
active = m.dis.idomain.array[0, :, :].flatten()
mean = np.zeros_like(easting)        # zero mean to keep transforms simple later
var = np.ones_like(easting)
aa = np.ones_like(easting) * 1000.0  # variogram range [length units]
anis = np.ones_like(easting) * 3.0
bearing = np.ones_like(easting) * 45.0

transform = "none"        # 0: natural, 1: log
variogram_type = "exp"    # 1:spher 2:exp 3:gauss 4:pow
power = 1.0               # only used when variogram_type == 'pow'
nnode = easting.size
nnode

## Demo 1 — `fieldgen2d_sva_iid` reproduces `fieldgen2d_sva`

`fill_stdnormal(nrow, ncol)` draws variates in exactly the same loop order
`fieldgen2d_sva` uses internally. So:

1. `initialize_randgen(seed)` + `fieldgen2d_sva(...)`

is bit-identical to:

2. `initialize_randgen(seed)` + `fill_stdnormal(nnode, nreal)` + `fieldgen2d_sva_iid(..., diid)`

In [ ]:
seed = 12345
nreal = 6

# (1) original path — let the Fortran RNG draw the variates internally
lib_a = PestUtilsLib()
lib_a.initialize_randgen(seed)
field_orig = lib_a.fieldgen2d_sva(
    easting, northing, area, active,
    mean, var, aa, anis, bearing,
    transform, variogram_type, power, nreal,
)

# (2) iid path — draw the variates first with fill_stdnormal, then pass them in
lib_b = PestUtilsLib()
lib_b.initialize_randgen(seed)
diid = lib_b.fill_stdnormal(nnode, nreal)
field_iid = lib_b.fieldgen2d_sva_iid(
    easting, northing, area, active,
    mean, var, aa, anis, bearing,
    transform, variogram_type, power, diid,
)

print("max abs difference:", np.max(np.abs(field_orig - field_iid)))
np.testing.assert_array_equal(field_orig, field_iid)
print("identical: True")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
r_orig = field_orig[:, 0].reshape((nrow, ncol))
r_iid = field_iid[:, 0].reshape((nrow, ncol))
diff = r_orig - r_iid
for ax, arr, title in zip(axes, [r_orig, r_iid, diff],
                          ["fieldgen2d_sva", "fieldgen2d_sva_iid", "difference"]):
    ax.set_aspect("equal")
    cb = ax.pcolormesh(Easting, Northing, arr)
    ax.set_title(title)
    plt.colorbar(cb, ax=ax, shrink=0.7)

## Demo 2 — drive the iid version with NumPy's RNG

The iid variant does not require `initialize_randgen` to have been called — you can
hand it any standard-normal array of the right shape. This is convenient if you
want to control the RNG from Python (e.g. for cross-platform reproducibility or
to integrate with an existing ensemble workflow).

In [ ]:
rng = np.random.default_rng(2026)
diid_np = rng.standard_normal(size=(nnode, nreal))

lib_c = PestUtilsLib()  # no initialize_randgen call
field_np = lib_c.fieldgen2d_sva_iid(
    easting, northing, area, active,
    mean, var, aa, anis, bearing,
    transform, variogram_type, power, diid_np,
)
print("field_np.shape:", field_np.shape)
print("field_np finite:", np.all(np.isfinite(field_np)))

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for k, ax in enumerate(axes.ravel()):
    r = field_np[:, k].reshape((nrow, ncol))
    ax.set_aspect("equal")
    cb = ax.pcolormesh(Easting, Northing, r)
    ax.set_title(f"realisation {k}")
    plt.colorbar(cb, ax=ax, shrink=0.7)
fig.suptitle("Realisations driven by numpy's default_rng")
fig.tight_layout()

## Demo 3 — same `diid`, fresh `PestUtilsLib`, identical field

Because `fieldgen2d_sva_iid` consumes only the variates you pass in, the result is
fully deterministic in `diid` — no hidden RNG state. Two fresh library instances
with the same `diid` produce bit-identical fields.

In [ ]:
def run_iid(diid):
    lib_local = PestUtilsLib()
    return lib_local.fieldgen2d_sva_iid(
        easting, northing, area, active,
        mean, var, aa, anis, bearing,
        transform, variogram_type, power, diid,
    )

f1 = run_iid(diid_np)
f2 = run_iid(diid_np)
np.testing.assert_array_equal(f1, f2)
print("two fresh libs, same diid -> bit-identical:", np.array_equal(f1, f2))

## Demo 4 — linearity of the field map for `transtype=0`, `mean=0`

With `transtype="none"` (natural-space) and a zero mean, `fieldgen2d_sva_iid` is a
*linear* operator on `diid`: the field is a deterministic spatially-correlating
linear combination of the variates. So for any scalars `a`, `b` and any pair of
variate vectors `d1`, `d2`:

```
F(a*d1 + b*d2) == a*F(d1) + b*F(d2)
```

This linearity is what makes the iid form attractive for sensitivity analysis and
ensemble linear-algebra tricks (e.g. recombining realisations without
re-running the convolution).

In [ ]:
rng = np.random.default_rng(7)
d1 = rng.standard_normal(size=(nnode, 1))
d2 = rng.standard_normal(size=(nnode, 1))
a, b = 0.7, -1.3

lib_d = PestUtilsLib()
common = dict(
    ec=easting, nc=northing, area=area, active=active,
    mean=mean, var=var, aa=aa, anis=anis, bearing=bearing,
    transtype=transform, avetype=variogram_type, power=power,
)
f_d1 = lib_d.fieldgen2d_sva_iid(diid=d1, **common)
f_d2 = lib_d.fieldgen2d_sva_iid(diid=d2, **common)
f_comb = lib_d.fieldgen2d_sva_iid(diid=a * d1 + b * d2, **common)

lhs = f_comb
rhs = a * f_d1 + b * f_d2
print("max abs difference:", np.max(np.abs(lhs - rhs)))
np.testing.assert_allclose(lhs, rhs, rtol=1e-10, atol=1e-10)
print("linear in diid: True")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
panels = [
    ("F(d1)", f_d1[:, 0].reshape((nrow, ncol))),
    ("F(d2)", f_d2[:, 0].reshape((nrow, ncol))),
    ("F(a*d1 + b*d2)", f_comb[:, 0].reshape((nrow, ncol))),
]
for ax, (title, arr) in zip(axes, panels):
    ax.set_aspect("equal")
    cb = ax.pcolormesh(Easting, Northing, arr)
    ax.set_title(title)
    plt.colorbar(cb, ax=ax, shrink=0.7)

## Wrap up

Key points:

- `fill_stdnormal` + `fieldgen2d_sva_iid` reproduces `fieldgen2d_sva` exactly when
  given the same seed.
- `fieldgen2d_sva_iid` does not require `initialize_randgen` — bring your own RNG.
- For `transtype=0` (natural space) with zero mean, the field is a linear function
  of the input variates, which lets you recombine realisations cheaply.